# Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# import networkx as nx
import rich
from scipy.optimize import minimize
from tqdm.auto import tqdm

# Main

## 1. Подготовка данных

## Коропоративные облигации

In [2]:

# ============================================================
#  ОТБОР КОРПОРАТИВНЫХ ОБЛИГАЦИЙ ДЛЯ ПОРТФЕЛЯ
#  Источник данных: Cbonds (выгрузка Korp.csv)
#  Горизонт инвестирования H = 3 года
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# БЛОК 0: ЗАГРУЗКА И ПРИВЕДЕНИЕ ТИПОВ
# ------------------------------------------------------------

H = 3.0  # инвестиционный горизонт, лет

df = pd.read_csv("Корп.csv", encoding="utf-8", sep=",", quotechar='"')

# Парсим числа с русской локалью (пробел=тысячи, запятая=десятичная)
def clean_num(s):
    return pd.to_numeric(
        s.astype(str).str.strip()
         .str.replace("\xa0", "", regex=False)
         .str.replace(" ",    "", regex=False)
         .str.replace(",",    ".", regex=False),
        errors="coerce"
    )

num_cols = [
    "Индикативная доходность, %",
    "Ликвидность",
    "Дюрация, дней",
    "Модифицированная дюрация",
    "Срок до погашения, лет",
    "Текущий купон, %",
    "Индикативная цена, %",
    "G-spread, бп",
]
for c in num_cols:
    df[c] = clean_num(df[c])

print(f"Исходно строк: {len(df)}")

# ------------------------------------------------------------
# БЛОК 1: ФИЛЬТРАЦИЯ
# Каждый фильтр выводит сколько строк осталось
# ------------------------------------------------------------

# 1.1 Только бумаги с ISIN (отсекает ЦФА и незарегистрированные)
df = df[df["ISIN"].notna() & (df["ISIN"].str.strip() != "")]
print(f"  после фильтра ISIN:              {len(df)}")

# 1.2 Только фиксированная процентная ставка
#     Флоатеры (КС+, RUONIA+) исключаем: YTM и дюрацию по ним
#     корректно посчитать нельзя
df = df[df["Тип ставки"].str.strip() == "Фиксированная"]
print(f"  после фильтра фикс. ставки:      {len(df)}")

# 1.3 Только купонные облигации
#     Дисконтные Sber CIB / структурные продукты — не для портфеля
df = df[df["Вид облигации"].str.strip() == "Купонные"]
print(f"  после фильтра купонных:          {len(df)}")

# 1.4 Ликвидность >= 3 (по шкале Cbonds)
#     Уровни 1-2 = единичные сделки в месяц, нельзя выйти по цене
df = df[df["Ликвидность"] >= 3]
print(f"  после фильтра ликвидности >= 3:  {len(df)}")

# 1.5 Рейтинг АКРА: только A(RU) и выше
#     Бумаги без рейтинга несут неизвестный кредитный риск
rating_map = {
    "AAA(RU)": 6,
    "AA+(RU)": 5,
    "AA(RU)":  4,
    "AA-(RU)": 3,
    "A+(RU)":  2,
    "A(RU)":   1,
}
df["rating_score"] = df["Рейтинг эмитента АКРА"].str.strip().map(rating_map)
df = df[df["rating_score"].notna()]
print(f"  после фильтра рейтинга >= A(RU): {len(df)}")

# 1.6 Убираем call-оферты
#     Эмитент с call может досрочно погасить бумагу при снижении
#     ставок — именно тогда, когда нам это невыгодно
df = df[df["Оферта (call)"].isna() |
        (df["Оферта (call)"].astype(str).str.strip() == "")]
print(f"  после фильтра call-опционов:     {len(df)}")

# 1.7 YTM в диапазоне 5–35% (убирает аномалии и технические строки)
df = df[df["Индикативная доходность, %"].notna()]
df = df[
    (df["Индикативная доходность, %"] > 5) &
    (df["Индикативная доходность, %"] < 35)
]
print(f"  после фильтра YTM 5-35%:         {len(df)}")

# 1.8 Дюрация > 180 дней (не берём «почти погашённые»)
df = df[df["Дюрация, дней"].notna() & (df["Дюрация, дней"] > 180)]
print(f"  после фильтра дюрации > 180 дн:  {len(df)}")

print(f"\n✅ После всех фильтров: {len(df)} бумаг\n")

# ------------------------------------------------------------
# БЛОК 2: СКОРИНГ
#
# Веса критериев обоснованы иерархией целей портфеля:
#   40% YTM     — главная причина включать корпораты вместо ОФЗ
#   30% Рейтинг — контроль кредитного риска
#   20% Ликв.   — исполнимость в реальных торгах
#   10% Дюрация — подстройка к горизонту (точная — через Solver)
#
# Метод: ранги (меньший ранг = лучше)
# ------------------------------------------------------------

df["dur_years"] = df["Дюрация, дней"] / 365.0
df["dur_dist"]  = (df["dur_years"] - H).abs()

df["rank_ytm"]    = df["Индикативная доходность, %"].rank(ascending=False, method="min")
df["rank_rating"] = df["rating_score"].rank(ascending=False, method="min")
df["rank_liq"]    = df["Ликвидность"].rank(ascending=False, method="min")
df["rank_dur"]    = df["dur_dist"].rank(ascending=True,  method="min")

df["score"] = (0.40 * df["rank_ytm"] +
               0.30 * df["rank_rating"] +
               0.20 * df["rank_liq"] +
               0.10 * df["rank_dur"])
# Меньше score = лучше

# ------------------------------------------------------------
# БЛОК 3: ДЕДУПЛИКАЦИЯ ПО ЭМИТЕНТУ
# Оставляем лучший выпуск каждого эмитента по скору
# ------------------------------------------------------------

df_dedup = (
    df.sort_values("score")
      .drop_duplicates(subset="Эмитент", keep="first")
      .copy()
)
print(f"Уникальных эмитентов после фильтров: {len(df_dedup)}")

# ------------------------------------------------------------
# БЛОК 4: ФИНАЛЬНЫЙ ОТБОР С ЛИМИТОМ НА СЕКТОР
#
# Лимит: не более 2 бумаг из одного сектора
# Это обеспечивает секторальную диверсификацию
# ------------------------------------------------------------

SECTOR_LIMIT  = 2   # макс. бумаг из одного сектора
PORTFOLIO_N   = 30  # целевое число бумаг

selected      = []
sector_counts = {}

for _, row in df_dedup.sort_values("score").iterrows():
    sector = row["Отрасль"]
    if sector_counts.get(sector, 0) >= SECTOR_LIMIT:
        continue
    selected.append(row)
    sector_counts[sector] = sector_counts.get(sector, 0) + 1
    if len(selected) == PORTFOLIO_N:
        break

final = pd.DataFrame(selected)

# ------------------------------------------------------------
# БЛОК 5: ВЫВОД РЕЗУЛЬТАТОВ
# ------------------------------------------------------------

print("\n" + "="*70)
print("  ФИНАЛЬНЫЙ СПИСОК КОРПОРАТИВНЫХ ОБЛИГАЦИЙ (кандидаты в портфель)")
print("="*70)

for i, (_, r) in enumerate(final.iterrows(), 1):
    print(f"  {i:2d}. {r['Эмитент']:30s} "
          f"| {r['Рейтинг эмитента АКРА']:10s} "
          f"| YTM {r['Индикативная доходность, %']:5.2f}% "
          f"| Дюр {r['dur_years']:.2f} л. "
          f"| Ликв {int(r['Ликвидность']):d} "
          f"| {r['Отрасль']}")

print("\n--- Сводные характеристики финального списка ---")
print(f"  Средняя YTM:          {final['Индикативная доходность, %'].mean():.2f}%")
print(f"  Мин / Макс YTM:       {final['Индикативная доходность, %'].min():.2f}% / "
                               f"{final['Индикативная доходность, %'].max():.2f}%")
print(f"  Средняя дюрация:      {final['dur_years'].mean():.2f} лет")
print(f"  Уникальных секторов:  {final['Отрасль'].nunique()}")
print(f"  Мин. рейтинг:         {final['Рейтинг эмитента АКРА'].map(rating_map).min()} "
      f"({final.loc[final['Рейтинг эмитента АКРА'].map(rating_map).idxmin(), 'Рейтинг эмитента АКРА']})")

print("\n--- Распределение по секторам ---")
print(final["Отрасль"].value_counts().to_string())

# ------------------------------------------------------------
# БЛОК 6: СОХРАНЕНИЕ
# ------------------------------------------------------------

# Финальные 10 кандидатов
final[[
    "Бумага", "ISIN", "Эмитент", "Отрасль",
    "Рейтинг эмитента АКРА", "rating_score",
    "Индикативная доходность, %", "dur_years",
    "Ликвидность", "score"
]].to_csv("corporate_shortlist.csv", index=False, encoding="utf-8-sig")

# Все 286 отфильтрованных — для ручной замены при необходимости
df.sort_values("score")[[
    "Бумага", "ISIN", "Эмитент", "Отрасль",
    "Рейтинг эмитента АКРА", "rating_score",
    "Индикативная доходность, %", "dur_years",
    "Ликвидность", "score"
]].to_csv("corporate_filtered_all.csv", index=False, encoding="utf-8-sig")

print("\n✅ Сохранено:")
print("   corporate_shortlist.csv    — 10 финальных кандидатов")
print("   corporate_filtered_all.csv — все 286 отфильтрованных бумаг (резерв)")


Исходно строк: 3168
  после фильтра ISIN:              2683
  после фильтра фикс. ставки:      2683
  после фильтра купонных:          1996
  после фильтра ликвидности >= 3:  704
  после фильтра рейтинга >= A(RU): 601
  после фильтра call-опционов:     514
  после фильтра YTM 5-35%:         405
  после фильтра дюрации > 180 дн:  286

✅ После всех фильтров: 286 бумаг

Уникальных эмитентов после фильтров: 59

  ФИНАЛЬНЫЙ СПИСОК КОРПОРАТИВНЫХ ОБЛИГАЦИЙ (кандидаты в портфель)
   1. Sber CIB                       | AAA(RU)    | YTM 32.18% | Дюр 2.51 л. | Ликв 4 | Финансовые рынки
   2. РЖД                            | AAA(RU)    | YTM 15.68% | Дюр 2.39 л. | Ликв 7 | Железнодорожный транспорт
   3. МТС                            | AAA(RU)    | YTM 16.08% | Дюр 0.93 л. | Ликв 7 | Связь и телекоммуникация
   4. МБЭС                           | AAA(RU)    | YTM 15.62% | Дюр 2.05 л. | Ликв 7 | Банки
   5. Россети                        | AAA(RU)    | YTM 15.31% | Дюр 3.36 л. | Ликв 7 | Электроэн

## ОФЗ

In [3]:

# ============================================================
#  ОТБОР ОФЗ ДЛЯ ПОРТФЕЛЯ — СТРАТЕГИЯ BARBELL
#  Источник данных: Cbonds (выгрузка OFZ.csv)
#  Горизонт инвестирования H = 3 года
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# БЛОК 0: ЗАГРУЗКА И ПРИВЕДЕНИЕ ТИПОВ
# ------------------------------------------------------------

H = 3.0  # <- МЕНЯЙ ЗДЕСЬ горизонт в годах

def clean_num(s):
    """Парсит числа с русской локалью: пробел=тысячи, запятая=десятичная."""
    return pd.to_numeric(
        s.astype(str).str.strip()
         .str.replace("\xa0", "", regex=False)
         .str.replace(" ",    "", regex=False)
         .str.replace(",",    ".", regex=False),
        errors="coerce"
    )

df = pd.read_csv("ОФЗ.csv", encoding="utf-8", sep=",", quotechar='"')

num_cols = [
    "Индикативная доходность, %",
    "Ликвидность",
    "Дюрация, дней",
    "Модифицированная дюрация",
    "Индикативная цена, %",
    "Текущий купон, %",
]
for c in num_cols:
    df[c] = clean_num(df[c])

df["dur_years"] = df["Дюрация, дней"] / 365.0
df["серия"]     = df["Бумага"].str.extract(r"(\d{5})")

print(f"Исходно: {len(df)} ОФЗ")

# ------------------------------------------------------------
# БЛОК 1: ФИЛЬТРАЦИЯ
# ------------------------------------------------------------

# 1.1 Только серия 26xxx — ОФЗ-ПД (постоянный доход, фикс. купон)
#     Исключаем:
#       46xxx (ОФЗ-АД) — амортизируемые, дюрация считается иначе
#       52xxx (ОФЗ-ИН) — линкеры к инфляции, YTM в реальном выражении
#       36xxx (ГСО)    — не торгуются для частных инвесторов
df = df[df["серия"].str.startswith("26", na=False)].copy()
print(f"  после фильтра серии 26xxx:        {len(df)}")

# 1.2 Ликвидность >= 6 (у ОФЗ шкала 1-8, ставим высокую планку)
df = df[df["Ликвидность"] >= 6]
print(f"  после фильтра ликвидности >= 6:   {len(df)}")

# 1.3 YTM в диапазоне 5-20% (убирает технические строки без котировок)
df = df[df["Индикативная доходность, %"].notna()]
df = df[
    (df["Индикативная доходность, %"] > 5) &
    (df["Индикативная доходность, %"] < 20)
]
print(f"  после фильтра YTM 5-20%:          {len(df)}")

# 1.4 Дюрация должна быть заполнена и > 0
df = df[df["Дюрация, дней"].notna() & (df["Дюрация, дней"] > 0)]
print(f"  после фильтра дюрации > 0:        {len(df)}")

print(f"\n✅ После всех фильтров: {len(df)} ОФЗ\n")

# ------------------------------------------------------------
# БЛОК 2: ОТБОР ПО СТРАТЕГИИ BARBELL
#
# Три зоны дюрации вокруг горизонта H:
#   Короткая : D < H - 0.5        — ликвидность + якорь
#   Средняя  : H - 0.5 <= D <= H + 0.5  — стабилизация дюрации
#   Длинная  : D > H + 0.5        — максимизация выпуклости
#
# Из каждой зоны берём 2 самые ликвидные бумаги.
# Barbell даёт бОльшую выпуклость, чем bullet при той же дюрации:
# при любом сдвиге ставок цена портфеля падает меньше / растёт больше.
# ------------------------------------------------------------

zone1 = df[df["dur_years"] < H - 0.5].nlargest(2, "Ликвидность")
zone2 = df[(df["dur_years"] >= H - 0.5) &
           (df["dur_years"] <= H + 0.5)].nlargest(2, "Ликвидность")
zone3 = df[df["dur_years"] > H + 0.5].nlargest(2, "Ликвидность")

ofz_selected = (
    pd.concat([zone1, zone2, zone3])
      .drop_duplicates(subset="ISIN")
      .sort_values("dur_years")
)

# ------------------------------------------------------------
# БЛОК 3: ВЫВОД РЕЗУЛЬТАТОВ
# ------------------------------------------------------------

print("=" * 80)
print("  ФИНАЛЬНЫЕ ОФЗ (6 бумаг, barbell)")
print("=" * 80)
print(f"  {'Зона':<10} {'Бумага':<45} {'YTM':>6}  {'Дюр':>7}  {'Ликв':>5}")
print("-" * 80)

for _, r in ofz_selected.iterrows():
    d = r["dur_years"]
    zone = "Короткая" if d < H - 0.5 else ("Средняя" if d <= H + 0.5 else "Длинная")
    print(f"  {zone:<10} {r['Бумага']:<45} "
          f"{r['Индикативная доходность, %']:>5.2f}%  "
          f"{d:>6.2f} л.  "
          f"{int(r['Ликвидность']):>4d}")

d_min = ofz_selected["dur_years"].min()
d_max = ofz_selected["dur_years"].max()
ok    = d_min < H < d_max

print(f"\n  Диапазон дюраций : {d_min:.2f} — {d_max:.2f} лет")
print(f"  Средняя YTM      : {ofz_selected['Индикативная доходность, %'].mean():.2f}%")
print(f"  Иммунизация H={H} : {'✅ ДА' if ok else '❌ НЕТ — скорректируй H или расширь зоны'}")

# ------------------------------------------------------------
# БЛОК 4: СОХРАНЕНИЕ
# ------------------------------------------------------------

cols_save = [
    "Бумага", "ISIN",
    "Индикативная доходность, %", "dur_years",
    "Ликвидность", "Индикативная цена, %", "Текущий купон, %",
]

ofz_selected[cols_save].to_csv(
    "ofz_shortlist.csv", index=False, encoding="utf-8-sig")

df[cols_save].sort_values("dur_years").to_csv(
    "ofz_filtered_all.csv", index=False, encoding="utf-8-sig")

print("\n✅ Сохранено:")
print("   ofz_shortlist.csv    — 6 ОФЗ для Solver")
print("   ofz_filtered_all.csv — все отфильтрованные ОФЗ (резерв)")


Исходно: 41 ОФЗ
  после фильтра серии 26xxx:        33
  после фильтра ликвидности >= 6:   32
  после фильтра YTM 5-20%:          32
  после фильтра дюрации > 0:        32

✅ После всех фильтров: 32 ОФЗ

  ФИНАЛЬНЫЕ ОФЗ (6 бумаг, barbell)
  Зона       Бумага                                           YTM      Дюр   Ликв
--------------------------------------------------------------------------------
  Короткая   Россия, 26219 (ОФЗ-ПД, SU26219RMFS4)          14.17%    0.53 л.     8
  Короткая   Россия, 26226 (ОФЗ-ПД, SU26226RMFS9)          14.30%    0.59 л.     8
  Средняя    Россия, 26224 (ОФЗ-ПД, SU26224RMFS4)          14.17%    2.86 л.     8
  Средняя    Россия, 26228 (ОФЗ-ПД, SU26228RMFS5)          14.36%    3.42 л.     8
  Длинная    Россия, 26241 (ОФЗ-ПД, SU26241RMFS8)          14.67%    4.77 л.     8
  Длинная    Россия, 26245 (ОФЗ-ПД, SU26245RMFS9)          14.76%    5.42 л.     8

  Диапазон дюраций : 0.53 — 5.42 лет
  Средняя YTM      : 14.40%
  Иммунизация H=3.0 : ✅ ДА

✅ Сохр

## Оптимизацния

In [4]:
# ============================================================
#  ОПТИМИЗАЦИЯ ВЕСОВ ОБЛИГАЦИОННОГО ПОРТФЕЛЯ
#  scipy SLSQP — аналог "Поиска решения" Excel
#
#  Входные файлы (генерируются bond_screener.py и ofz_screener.py):
#    ofz_shortlist.csv       — 6 ОФЗ
#    corporate_shortlist.csv — 10 корпоративных облигаций
#
#  Задача:
#    max  Σ w_i * YTM_i
#    s.t. Σ w_i * D_i  = H         (иммунизация дюрации)
#         Σ w_i        = 1         (полное инвестирование)
#         Σ w_i[ОФЗ]  ≥ 35%       (мин. доля госбумаг)
#         w_i          ≤ 15%       (макс. вес одной бумаги)
#         w_i[корп]    ≤ 10%       (макс. вес корпоратива)
#         w_i          ≥ 0         (нет коротких позиций)
# ============================================================

import pandas as pd
import numpy as np
from scipy.optimize import minimize

# ------------------------------------------------------------
# ПАРАМЕТРЫ
# ------------------------------------------------------------
H           = 3.0   # целевая дюрация (горизонт), лет
H_TOL       = 0.05  # допуск: дюрация попадёт в [H-tol, H+tol]
W_MAX       = 0.15  # макс. вес одной бумаги
W_MAX_CORP  = 0.10  # макс. вес одного корпоратива
W_MIN_OFZ   = 0.35  # мин. суммарный вес ОФЗ
# ------------------------------------------------------------

# ============================================================
# БЛОК 0: ЗАГРУЗКА И ОБЪЕДИНЕНИЕ ДАННЫХ
# ============================================================

ofz  = pd.read_csv("ofz_shortlist.csv",       encoding="utf-8-sig")
corp = pd.read_csv("corporate_shortlist.csv",  encoding="utf-8-sig")

ofz["тип"]  = "ОФЗ"
corp["тип"] = "Корпоратив"

ofz  = ofz.rename(columns={"Индикативная доходность, %": "ytm", "dur_years": "dur"})
corp = corp.rename(columns={"Индикативная доходность, %": "ytm", "dur_years": "dur"})

keep = ["Бумага", "ISIN", "ytm", "dur", "Ликвидность", "тип"]
df = pd.concat([ofz[keep], corp[keep]], ignore_index=True)

n       = len(df)
ytm_pct = df["ytm"].values          # в процентах (14.5, 20.0 ...)
ytm     = ytm_pct / 100             # в долях — для оптимизатора
dur     = df["dur"].values
is_ofz  = (df["тип"] == "ОФЗ").values
is_corp = ~is_ofz

print(f"Загружено бумаг: {n}  ({is_ofz.sum()} ОФЗ + {is_corp.sum()} корп.)")
print(f"YTM диапазон   : {ytm_pct.min():.2f}% — {ytm_pct.max():.2f}%")
print(f"Дюрация диапаз.: {dur.min():.2f} — {dur.max():.2f} лет\n")

# ============================================================
# БЛОК 1: ПОСТАНОВКА ЗАДАЧИ ОПТИМИЗАЦИИ
# ============================================================

def objective(w):
    """Минимизируем отрицательный YTM → максимизируем YTM."""
    return -np.dot(w, ytm)

constraints = [
    # 1. Сумма весов = 1
    {"type": "eq",
     "fun":  lambda w: np.sum(w) - 1.0,
     "jac":  lambda w: np.ones(n)},

    # 2a. Нижняя граница иммунизации: D_port >= H - tol
    {"type": "ineq",
     "fun":  lambda w: np.dot(w, dur) - (H - H_TOL),
     "jac":  lambda w: dur},

    # 2b. Верхняя граница иммунизации: D_port <= H + tol
    {"type": "ineq",
     "fun":  lambda w: (H + H_TOL) - np.dot(w, dur),
     "jac":  lambda w: -dur},

    # 3. Минимальная доля ОФЗ
    {"type": "ineq",
     "fun":  lambda w: np.dot(w, is_ofz.astype(float)) - W_MIN_OFZ,
     "jac":  lambda w: is_ofz.astype(float)},
]

# Индивидуальные границы весов
lb = np.zeros(n)
ub = np.where(is_corp, W_MAX_CORP, W_MAX)
bounds = list(zip(lb, ub))

# ============================================================
# БЛОК 2: ЗАПУСК ОПТИМИЗАЦИИ
# Несколько стартовых точек — берём лучшее решение
# ============================================================

best_result = None
starts = [
    np.ones(n) / n,                           # равные веса
    np.where(is_ofz, W_MIN_OFZ / is_ofz.sum(),
             (1 - W_MIN_OFZ) / is_corp.sum()), # равно внутри групп
    np.random.dirichlet(np.ones(n)),           # случайный старт
]

for w0 in starts:
    # Проецируем начальное приближение в допустимую область
    w0 = np.clip(w0, lb, ub)
    w0 = w0 / w0.sum()

    res = minimize(
        fun=objective,
        x0=w0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"ftol": 1e-10, "maxiter": 2000, "disp": False},
    )
    if res.success and (best_result is None or res.fun < best_result.fun):
        best_result = res

if best_result is None or not best_result.success:
    raise RuntimeError(f"Solver не нашёл решения: {best_result.message}")

w_opt = best_result.x
w_opt = np.where(w_opt < 1e-4, 0.0, w_opt)   # обнуляем < 0.01%
w_opt = w_opt / w_opt.sum()                   # нормируем до 1

print("✅ Оптимальное решение найдено!")

# ============================================================
# БЛОК 3: ВЫВОД РЕЗУЛЬТАТОВ
# ============================================================

df["вес_%"]      = w_opt * 100
df["вклад_ytm"]  = w_opt * ytm_pct
df["вклад_dur"]  = w_opt * dur

port_ytm = df["вклад_ytm"].sum()   # уже в %
port_dur = df["вклад_dur"].sum()
w_ofz_s  = df.loc[is_ofz,  "вес_%"].sum() / 100
w_corp_s = df.loc[is_corp, "вес_%"].sum() / 100
hhi      = np.sum(w_opt**2)

print()
print("=" * 80)
print(f"  ОПТИМАЛЬНЫЙ ПОРТФЕЛЬ  |  H = {H} лет  |  max YTM  (SLSQP)")
print("=" * 80)
print(f"  {'#':<3} {'Бумага':<42} {'Тип':<12} {'Вес':>7}  {'YTM':>6}  {'Дюр':>6}")
print("-" * 80)

active = df[df["вес_%"] > 0.01].sort_values("тип")
for i, (_, r) in enumerate(active.iterrows(), 1):
    print(f"  {i:<3} {str(r['Бумага'])[:41]:<42} {r['тип']:<12} "
          f"{r['вес_%']:>6.1f}%  "
          f"{r['ytm']:>5.2f}%  "
          f"{r['dur']:>5.2f} л.")

print("=" * 80)
print(f"\n  ХАРАКТЕРИСТИКИ ПОРТФЕЛЯ:")
print(f"  {'YTM портфеля:':<35} {port_ytm:.2f}%")
print(f"  {'Дюрация:':<35} {port_dur:.3f} лет  "
      f"(цель {H} ± {H_TOL})")
print(f"  {'Доля ОФЗ:':<35} {w_ofz_s*100:.1f}%  "
      f"(мин {W_MIN_OFZ*100:.0f}%)")
print(f"  {'Доля корпоративных:':<35} {w_corp_s*100:.1f}%")
print(f"  {'Активных позиций:':<35} {(df['вес_%'] > 0.01).sum()} из {n}")
print(f"  {'HHI (концентрация):':<35} {hhi:.4f}  "
      f"({'низкая' if hhi < 0.10 else 'умеренная' if hhi < 0.18 else 'высокая'})")

print(f"\n  ПРОВЕРКА ОГРАНИЧЕНИЙ:")
checks = [
    ("Σ весов = 1",
     abs(w_opt.sum() - 1) < 1e-4,
     f"{w_opt.sum():.6f}"),
    (f"Дюрация в [{H-H_TOL:.2f}, {H+H_TOL:.2f}] лет",
     abs(port_dur - H) <= H_TOL + 1e-4,
     f"{port_dur:.4f}"),
    (f"ОФЗ >= {W_MIN_OFZ*100:.0f}%",
     w_ofz_s >= W_MIN_OFZ - 1e-4,
     f"{w_ofz_s*100:.1f}%"),
    (f"Корпоратив <= {W_MAX_CORP*100:.0f}% (каждая)",
     all(w_opt[is_corp] <= W_MAX_CORP + 1e-4),
     ""),
    (f"Любая бумага <= {W_MAX*100:.0f}%",
     all(w_opt <= W_MAX + 1e-4),
     ""),
]
for label, flag, detail in checks:
    print(f"  {'✅' if flag else '❌'} {label:<38} {detail}")

# ============================================================
# БЛОК 4: СОХРАНЕНИЕ
# ============================================================

df[["Бумага", "ISIN", "тип", "вес_%", "ytm", "dur", "Ликвидность"]]\
    .rename(columns={
        "вес_%":       "Вес, %",
        "ytm":         "YTM, %",
        "dur":         "Дюрация, лет",
        "Ликвидность": "Ликвидность",
    })\
    .sort_values(["тип", "Вес, %"], ascending=[True, False])\
    .to_csv("portfolio_weights.csv", index=False, encoding="utf-8-sig")

print("\n✅ Сохранено: portfolio_weights.csv")


Загружено бумаг: 36  (6 ОФЗ + 30 корп.)
YTM диапазон   : 14.17% — 32.18%
Дюрация диапаз.: 0.53 — 5.42 лет

✅ Оптимальное решение найдено!

  ОПТИМАЛЬНЫЙ ПОРТФЕЛЬ  |  H = 3.0 лет  |  max YTM  (SLSQP)
  #   Бумага                                     Тип              Вес     YTM     Дюр
--------------------------------------------------------------------------------
  1   Sber CIB, CIB-СО-548                       Корпоратив     10.0%  32.18%   2.51 л.
  2   Россети, 001P-04R                          Корпоратив      7.9%  15.31%   3.36 л.
  3   Банк ВТБ (ПАО), Б-1-245                    Корпоратив     10.0%  25.35%   0.70 л.
  4   Атомэнергопром, 001P-10                    Корпоратив     10.0%  15.21%   3.67 л.
  5   АФК Система, 001P-11                       Корпоратив      7.1%  20.59%   1.32 л.
  6   Балтийский лизинг, БО-П15                  Корпоратив     10.0%  22.21%   0.96 л.
  7   Рольф, 001P-08                             Корпоратив     10.0%  24.35%   1.07 л.
  8   Россия, 2622

## Графики

In [6]:

# ============================================================
#  ВИЗУАЛИЗАЦИЯ ОБЛИГАЦИОННОГО ПОРТФЕЛЯ
#  Входной файл: portfolio_weights.csv
#  Выходные файлы: portfolio_weights_bar.png
#                  portfolio_ytm_contrib.png
#                  portfolio_map.png
# ============================================================

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import json

# ------------------------------------------------------------
# БЛОК 0: ЗАГРУЗКА ДАННЫХ
# ------------------------------------------------------------

df = pd.read_csv("portfolio_weights.csv", encoding="utf-8-sig")
df = df[df["Вес, %"] > 0.1].copy()
df["Имя"] = df["Бумага"].apply(lambda s: " ".join(str(s).split()[:2]))

COLOR = {"ОФЗ": "#4f9cf4", "Корпоратив": "#f4a24f"}
df["color"] = df["тип"].map(COLOR)

port_ytm = np.dot(df["Вес, %"] / 100, df["YTM, %"])
port_dur = np.dot(df["Вес, %"] / 100, df["Дюрация, лет"])
w_ofz    = df.loc[df["тип"] == "ОФЗ",        "Вес, %"].sum()
w_corp   = df.loc[df["тип"] == "Корпоратив", "Вес, %"].sum()

# ============================================================
# CHART 1: Горизонтальный бар — веса бумаг
# ============================================================

active = df.sort_values("Вес, %")

fig1 = go.Figure()
for typ, clr in COLOR.items():
    d = active[active["тип"] == typ]
    fig1.add_trace(go.Bar(
        x=d["Вес, %"],
        y=d["Имя"],
        orientation="h",
        name=typ,
        marker_color=clr,
        text=[f"  {w:.1f}%  |  YTM {y:.1f}%"
              for w, y in zip(d["Вес, %"], d["YTM, %"])],
        textposition="outside",
        textfont=dict(size=11),
        cliponaxis=False,
    ))

fig1.update_layout(
    title={"text": (
        "Структура портфеля по бумагам<br>"
        f"<span style='font-size:14px;font-weight:normal;'>"
        f"YTM: {port_ytm:.2f}%  |  Дюрация: {port_dur:.2f} л.  |  "
        f"ОФЗ {w_ofz:.0f}% / Корп {w_corp:.0f}%</span>"
    )},
    barmode="stack",
    legend=dict(orientation="h", yanchor="bottom", y=1.14,
                xanchor="center", x=0.5),
    margin=dict(t=130, r=160, b=60, l=160),
    xaxis=dict(range=[0, active["Вес, %"].max() * 1.6]),
)
fig1.update_xaxes(title_text="Вес, %")
fig1.update_yaxes(tickfont=dict(size=12))

fig1.write_image("portfolio_weights_bar.png")
with open("portfolio_weights_bar.png.meta.json", "w") as f:
    json.dump({"caption": "Структура портфеля: вес и YTM каждой бумаги"}, f)
print("✅ portfolio_weights_bar.png")

# ============================================================
# CHART 2: Горизонтальный бар — вклад каждой бумаги в YTM
# ============================================================

df["вклад"] = df["Вес, %"] / 100 * df["YTM, %"]
contrib = df.sort_values("вклад")

fig2 = go.Figure()
for typ, clr in COLOR.items():
    d = contrib[contrib["тип"] == typ]
    fig2.add_trace(go.Bar(
        x=d["вклад"],
        y=d["Имя"],
        orientation="h",
        name=typ,
        marker_color=clr,
        text=[f"  {v:.2f} п.п." for v in d["вклад"]],
        textposition="outside",
        textfont=dict(size=11),
        cliponaxis=False,
    ))

# Вертикаль итогового YTM — без перекрытия через add_annotation
fig2.add_vline(x=port_ytm, line_dash="dot", line_color="#888", line_width=2)
fig2.add_annotation(
    x=port_ytm, y=1.04, xref="x", yref="paper",
    text=f"YTM = {port_ytm:.2f}%",
    showarrow=False, font=dict(size=12, color="#555"),
    xanchor="left", bgcolor="rgba(255,255,255,0.85)", borderpad=3
)

fig2.update_layout(
    title={"text": (
        "Вклад каждой бумаги в YTM портфеля<br>"
        f"<span style='font-size:14px;font-weight:normal;'>"
        f"Итого: {port_ytm:.2f}%</span>"
    )},
    barmode="stack",
    legend=dict(orientation="h", yanchor="bottom", y=1.14,
                xanchor="center", x=0.5),
    margin=dict(t=130, r=160, b=60, l=160),
    xaxis=dict(range=[0, contrib["вклад"].max() * 1.65]),
)
fig2.update_xaxes(title_text="Вклад в YTM, п.п.")
fig2.update_yaxes(tickfont=dict(size=12))

fig2.write_image("portfolio_ytm_contrib.png")
with open("portfolio_ytm_contrib.png.meta.json", "w") as f:
    json.dump({"caption": "Вклад бумаг в итоговую доходность портфеля"}, f)
print("✅ portfolio_ytm_contrib.png")

# ============================================================
# CHART 3: Пузырьковый scatter — дюрация vs YTM
#
# Ключевой приём против наезда текста:
#   mode="markers" — текст убран из трейсов
#   add_annotation() для каждой точки с ручными ax/ay смещениями
# ============================================================

# ax = сдвиг вправо/влево в пикселях от точки
# ay = сдвиг вниз/вверх (положительный = вниз, отрицательный = вверх)
OFFSETS = {
    "ВТБ Б-1-245":              (-10, -26),
    "Банк ВТБ":                 (-10, -26),
    "АФК Система,":             ( 12, -26),
    "МТС, 002Р-07":             (-55, -26),
    "Роснефть, 002P-04":        ( -8,  28),
    "МБЭС, 002P-04":            ( 12,  30),
    "РЖД, 001P-41R":            ( 12, -26),
    "Россети, 001P-04R":        ( 12, -26),
    "Атомэнергопром, 001P-10":  ( 12,  30),
    "Россия, 26228":            (-10,  30),
    "Россия, 26241":            (-62, -26),
    "Россия, 26245":            ( 12, -26),
}

fig3 = go.Figure()

for typ, clr in COLOR.items():
    d = df[df["тип"] == typ]
    fig3.add_trace(go.Scatter(
        x=d["Дюрация, лет"],
        y=d["YTM, %"],
        name=typ,
        mode="markers",
        marker=dict(
            size=d["Вес, %"] * 3.8,
            color=clr,
            opacity=0.85,
            line=dict(width=1.5, color="white"),
        ),
        hovertemplate=(
            "<b>%{customdata}</b><br>"
            "Дюрация: %{x:.2f} л.<br>"
            "YTM: %{y:.2f}%<extra></extra>"
        ),
        customdata=d["Имя"],
    ))

# Аннотации с индивидуальными смещениями — гарантированно без наезда
for _, row in df.iterrows():
    ax, ay = OFFSETS.get(row["Имя"], (12, -26))
    fig3.add_annotation(
        x=row["Дюрация, лет"],
        y=row["YTM, %"],
        text=row["Имя"],
        showarrow=True,
        arrowhead=0,
        arrowwidth=1,
        arrowcolor="#ccc",
        ax=ax,
        ay=ay,
        font=dict(size=10),
        bgcolor="rgba(255,255,255,0.80)",
        borderpad=2,
        xanchor="center",
    )

# Вертикаль целевой дюрации
fig3.add_vline(x=port_dur, line_dash="dot", line_color="#888", line_width=2)
fig3.add_annotation(
    x=port_dur, y=0.97, xref="x", yref="paper",
    text=f"D = {port_dur:.2f} л.",
    showarrow=False, font=dict(size=12, color="#555"),
    xanchor="left", bgcolor="rgba(255,255,255,0.85)", borderpad=3
)

fig3.update_layout(
    title={"text": (
        "Карта портфеля: дюрация vs доходность<br>"
        "<span style='font-size:14px;font-weight:normal;'>"
        "Размер кружка = вес бумаги в портфеле</span>"
    )},
    legend=dict(orientation="h", yanchor="bottom", y=1.14,
                xanchor="center", x=0.5),
    margin=dict(t=130, r=60, b=70, l=70),
)
fig3.update_xaxes(
    title_text="Дюрация, лет",
    range=[-0.3, df["Дюрация, лет"].max() * 1.1]
)
fig3.update_yaxes(
    title_text="YTM, %",
    range=[df["YTM, %"].min() - 2.5, df["YTM, %"].max() + 4]
)

fig3.write_image("portfolio_map.png")
with open("portfolio_map.png.meta.json", "w") as f:
    json.dump({"caption": "Карта риска: дюрация vs доходность"}, f)
print("✅ portfolio_map.png")


✅ portfolio_weights_bar.png
✅ portfolio_ytm_contrib.png
✅ portfolio_map.png


In [7]:

# ============================================================
#  ВИЗУАЛИЗАЦИИ ХОДА РАБОТЫ — от сырых данных до портфеля
#  Входные файлы: OFZ.csv, Korp.csv, portfolio_weights.csv
#  Выходные файлы: vis_01_universe.png  — исходный юниверс
#                  vis_02_funnel_corp.png — воронка корпоратов
#                  vis_03_funnel_ofz.png  — воронка ОФЗ
#                  vis_04_scoring.png     — скоринг кандидатов
#                  vis_05_constraints.png — таблица ограничений
# ============================================================

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import json

# ============================================================
# БЛОК 0: ДАННЫЕ
# ============================================================

# --- 0.1 Шаги воронки корпоратов (заполни из лога bond_screener.py) ---
FUNNEL_CORP = [
    ("Исходный список",        3168),
    ("Есть ISIN",              2683),
    ("Купонные",               1996),
    ("Ликвидность >= 3",        704),
    ("Рейтинг >= A(RU)",        601),
    ("Без call-оферты",         514),
    ("YTM 5-35%",               405),
    ("Дюрация > 180 дней",      286),
]

# --- 0.2 Шаги воронки ОФЗ (заполни из лога ofz_screener.py) ---
FUNNEL_OFZ = [
    ("Исходный список",          41),
    ("Серия 26xxx (ОФЗ-ПД)",     33),
    ("Ликвидность >= 6",         32),
    ("YTM 5-20%",                32),
    ("Дюрация > 0",              32),
]

# --- 0.3 Топ-10 корпоративных кандидатов (заполни из bond_screener.py) ---
TOP_CORP = pd.DataFrame({
    "Эмитент":  ["РЖД","МТС","МБЭС","Россети","Банк ВТБ",
                 "Атомэнергопром","Газпром Нефть","Роснефть",
                 "АФК Система","Банк ПСБ"],
    "YTM":      [15.68,16.08,15.62,15.31,25.35,
                 15.21,15.03,15.36,20.59,14.99],
    "Рейтинг":  ["AAA","AAA","AAA","AAA","AAA",
                 "AAA","AAA","AAA","AA-","AAA"],
    "Дюр":      [2.39,0.93,2.05,3.36,0.70,
                 3.67,3.22,0.93,1.32,2.12],
    "Скор":     [49,53,56,62,64,69,74,77,81,82],
})

# --- 0.4 Ограничения оптимизатора ---
CONSTRAINTS = pd.DataFrame({
    "Параметр": [
        "Целевая дюрация H",
        "Допуск дюрации",
        "Макс. вес одной бумаги",
        "Макс. вес корпоратива",
        "Мин. доля ОФЗ",
        "Мин. рейтинг",
        "Тип ставки",
        "Ликвидность (корп.)",
        "Ликвидность (ОФЗ)",
        "Call-оферты",
    ],
    "Значение": [
        "3.0 года", "+/- 0.05 лет", "15%", "10%", "35%",
        "A(RU) по АКРА", "Только фикс.", ">= 3", ">= 6", "Исключены",
    ],
    "Обоснование": [
        "Горизонт инвестирования",
        "Допуск на погрешность Solver",
        "Диверсификация по бумагам",
        "Диверсификация по эмитентам",
        "Кредитное качество портфеля",
        "Ограничение кредитного риска",
        "YTM должна быть известна заранее",
        "Исполнимость заявки в торгах",
        "ОФЗ — высоколиквидный рынок",
        "Защита от реинвестиционного риска",
    ],
})

COLOR_CORP = "#f4a24f"
COLOR_OFZ  = "#4f9cf4"

# ============================================================
# CHART 1: Исходный юниверс — ОФЗ vs Корпоративные
# ============================================================

ofz_count  = FUNNEL_OFZ[0][1]
corp_count = FUNNEL_CORP[0][1]

fig1 = go.Figure()
fig1.add_trace(go.Bar(
    x=["ОФЗ<br>(OFZ.csv)", "Корпоративные<br>(Korp.csv)"],
    y=[ofz_count, corp_count],
    marker_color=[COLOR_OFZ, COLOR_CORP],
    width=0.4,
    text=[f"<b>{ofz_count}</b>", f"<b>{corp_count:,}</b>"],
    textposition="inside",
    textfont=dict(size=20, color="white"),
))

fig1.add_annotation(
    x="ОФЗ<br>(OFZ.csv)", y=-310,
    text="Серии 26xxx / 46xxx / 52xxx",
    showarrow=False, font=dict(size=11, color="#666"), xanchor="center"
)
fig1.add_annotation(
    x="Корпоративные<br>(Korp.csv)", y=-310,
    text="Весь корп. сектор Cbonds",
    showarrow=False, font=dict(size=11, color="#666"), xanchor="center"
)

fig1.update_layout(
    title={"text": (
        "Исходный юниверс данных<br>"
        "<span style='font-size:14px;font-weight:normal;'>"
        f"Суммарно {ofz_count + corp_count:,} выпусков из двух файлов"
        "</span>"
    )},
    showlegend=False,
    margin=dict(t=110, b=120, l=80, r=80),
    yaxis=dict(range=[-420, corp_count * 1.15]),
)
fig1.update_xaxes(title_text="")
fig1.update_yaxes(title_text="Кол-во выпусков")

fig1.write_image("vis_01_universe.png")
with open("vis_01_universe.png.meta.json", "w") as f:
    json.dump({"caption": f"Исходный юниверс: {ofz_count} ОФЗ и {corp_count:,} корп. облигаций"}, f)
print("OK vis_01_universe.png")

# ============================================================
# CHART 2: Воронка фильтрации корпоратов
# Горизонтальные убывающие бары — текст вынесен наружу
# ============================================================

labels_c = [s for s, _ in FUNNEL_CORP]
vals_c   = [v for _, v in FUNNEL_CORP]
pct_c    = [f"{v / vals_c[0] * 100:.0f}%" for v in vals_c]
idx_c    = list(range(len(labels_c)))

fig2 = go.Figure()
fig2.add_trace(go.Bar(
    y=idx_c,
    x=vals_c,
    orientation="h",
    marker=dict(
        color=vals_c,
        colorscale=[[0, "#fce4c4"], [1, COLOR_CORP]],
        showscale=False,
        line=dict(width=0),
    ),
    # Текст строго снаружи бара, с отступом
    text=[f"  {v:,}  ({p})" for v, p in zip(vals_c, pct_c)],
    textposition="outside",
    textfont=dict(size=12),
    cliponaxis=False,
))

fig2.update_layout(
    title={"text": (
        "Фильтрация корпоративных облигаций<br>"
        f"<span style='font-size:14px;font-weight:normal;'>"
        f"{vals_c[0]:,} → {vals_c[-1]} бумаг  "
        f"({vals_c[-1]/vals_c[0]*100:.0f}% от исходного списка)"
        "</span>"
    )},
    # Большой правый отступ — место для текста за баром
    margin=dict(t=110, l=200, r=200, b=60),
    xaxis=dict(range=[0, max(vals_c) * 1.55]),
    yaxis=dict(
        tickmode="array",
        tickvals=idx_c,
        ticktext=labels_c,
        tickfont=dict(size=12),
        autorange="reversed",
    ),
)
fig2.update_xaxes(title_text="Осталось выпусков")

fig2.write_image("vis_02_funnel_corp.png")
with open("vis_02_funnel_corp.png.meta.json", "w") as f:
    json.dump({"caption": f"Фильтрация корп. облигаций: {vals_c[0]:,} → {vals_c[-1]}"}, f)
print("OK vis_02_funnel_corp.png")

# ============================================================
# CHART 3: Воронка фильтрации ОФЗ
# ============================================================

labels_o = [s for s, _ in FUNNEL_OFZ]
vals_o   = [v for _, v in FUNNEL_OFZ]
pct_o    = [f"{v / vals_o[0] * 100:.0f}%" for v in vals_o]
idx_o    = list(range(len(labels_o)))

fig3 = go.Figure()
fig3.add_trace(go.Bar(
    y=idx_o,
    x=vals_o,
    orientation="h",
    marker=dict(
        color=vals_o,
        colorscale=[[0, "#c8e0fa"], [1, COLOR_OFZ]],
        showscale=False,
        line=dict(width=0),
    ),
    text=[f"  {v}  ({p})" for v, p in zip(vals_o, pct_o)],
    textposition="outside",
    textfont=dict(size=13),
    cliponaxis=False,
))

fig3.update_layout(
    title={"text": (
        "Фильтрация ОФЗ<br>"
        f"<span style='font-size:14px;font-weight:normal;'>"
        f"{vals_o[0]} → {vals_o[-1]} бумаг  "
        "(серии 46xxx, 52xxx, 36xxx исключены)"
        "</span>"
    )},
    margin=dict(t=110, l=200, r=140, b=60),
    xaxis=dict(range=[0, max(vals_o) * 1.4]),
    yaxis=dict(
        tickmode="array",
        tickvals=idx_o,
        ticktext=labels_o,
        tickfont=dict(size=13),
        autorange="reversed",
    ),
)
fig3.update_xaxes(title_text="Осталось выпусков")

fig3.write_image("vis_03_funnel_ofz.png")
with open("vis_03_funnel_ofz.png.meta.json", "w") as f:
    json.dump({"caption": f"Фильтрация ОФЗ: {vals_o[0]} → {vals_o[-1]} ОФЗ-ПД"}, f)
print("OK vis_03_funnel_ofz.png")

# ============================================================
# CHART 4: Скоринг корпоративных кандидатов
# Горизонтальный бар, текст строго за баром
# ============================================================

tc = TOP_CORP.sort_values("Скор").reset_index(drop=True)
n  = len(tc)
# Топ-2 (с наименьшим скором) — самый тёмный цвет
bar_colors = [COLOR_CORP if i >= n - 2 else "#f4c48f" for i in range(n)]

fig4 = go.Figure()
fig4.add_trace(go.Bar(
    y=tc["Эмитент"],
    x=tc["Скор"],
    orientation="h",
    marker_color=bar_colors,
    # Текст: YTM | Дюрация | Рейтинг — с пробелом перед, чтобы не примыкал к бару
    text=[
        f"  YTM {y:.1f}%  |  Дюр {d:.1f} л.  |  {r}"
        for y, d, r in zip(tc["YTM"], tc["Дюр"], tc["Рейтинг"])
    ],
    textposition="outside",
    textfont=dict(size=11),
    cliponaxis=False,
))

# Лёгкая подсветка зоны barbell-кандидатов
fig4.add_hrect(
    y0=n - 2 - 0.4, y1=n - 0.6,
    fillcolor="rgba(244,162,79,0.10)",
    line_width=1, line_color="rgba(244,162,79,0.5)",
)
fig4.add_annotation(
    x=tc["Скор"].max() * 1.52, y=n - 1.5,
    text="→ в портфель", showarrow=False,
    font=dict(size=11, color=COLOR_CORP), xanchor="right"
)

fig4.update_layout(
    title={"text": (
        "Скоринг корпоративных кандидатов<br>"
        "<span style='font-size:14px;font-weight:normal;'>"
        "Меньше баллов = лучше  (штраф за риск и неликвидность)"
        "</span>"
    )},
    margin=dict(t=110, l=160, r=280, b=60),
    xaxis=dict(range=[0, tc["Скор"].max() * 1.6]),
    yaxis=dict(tickfont=dict(size=12)),
)
fig4.update_xaxes(title_text="Скор (меньше = лучше)")

fig4.write_image("vis_04_scoring.png")
with open("vis_04_scoring.png.meta.json", "w") as f:
    json.dump({"caption": "Скоринг корп. кандидатов: топ-10 по качеству"}, f)
print("OK vis_04_scoring.png")

# ============================================================
# CHART 5: Таблица ограничений оптимизатора
# go.Table — самый надёжный способ, текст никуда не поедет
# ============================================================

fig5 = go.Figure(go.Table(
    # Ширины колонок в пикселях — подбираем под длину текста
    columnwidth=[190, 130, 300],
    header=dict(
        values=["<b>Параметр</b>", "<b>Значение</b>", "<b>Обоснование</b>"],
        fill_color=COLOR_OFZ,
        font=dict(color="white", size=13),
        align=["left", "center", "left"],
        height=38,
        line_color="white",
    ),
    cells=dict(
        values=[CONSTRAINTS[c].tolist() for c in CONSTRAINTS.columns],
        # Чередуем строки для читаемости
        fill_color=[
            ["#f0f6ff" if i % 2 == 0 else "white"
             for i in range(len(CONSTRAINTS))]
        ] * 3,
        font=dict(size=12),
        align=["left", "center", "left"],
        height=32,
        line_color="#dce8f5",
    ),
))

fig5.update_layout(
    title={"text": (
        "Ограничения портфельного оптимизатора (SLSQP)<br>"
        "<span style='font-size:14px;font-weight:normal;'>"
        "portfolio_optimizer.py  |  изменяй только БЛОК ПАРАМЕТРЫ"
        "</span>"
    )},
    margin=dict(t=110, l=30, r=30, b=30),
)

fig5.write_image("vis_05_constraints.png")
with open("vis_05_constraints.png.meta.json", "w") as f:
    json.dump({"caption": "Ограничения оптимизатора SLSQP"}, f)
print("OK vis_05_constraints.png")

print("\nВсе 5 файлов сохранены.")


OK vis_01_universe.png
OK vis_02_funnel_corp.png
OK vis_03_funnel_ofz.png
OK vis_04_scoring.png
OK vis_05_constraints.png

Все 5 файлов сохранены.


Цель портфеля
> Сформировать рублёвый облигационный портфель с горизонтом 3 года (февраль 2026 — февраль 2029), обеспечивающий доходность к погашению не ниже 18% годовых при умеренном кредитном риске, полной защите от процентного риска через иммунизацию и предсказуемых денежных потоках

Инвестиционные цели

| Параметр                    | Цель                        | Факт портфеля                  |
| --------------------------- | --------------------------- | ------------------------------ |
| Целевая доходность          | Выше ключевой ставки ЦБ РФ  | 16.76% YTM                     |
| Горизонт инвестирования     | 3 года                      | Дюрация 2.95 л. ≈ H            |
| Стратегия управления риском | Иммунизация дюрации         | Дюрация = H ± 0.05 ✅          |
| Сохранность капитала        | Отсутствие дефолтного риска | Только рейтинг ≥ A(RU) по АКРА |

Цель: максимизировать доходность к погашению рублёвого облигационного портфеля при горизонте 3 года, сохраняя умеренный кредитный риск и нейтрализуя процентный риск через иммунизацию дюрации.

Задачи:

Отобрать инструменты с рейтингом АКРА ≥ A(RU) и фиксированным купоном без call-опций

Обеспечить долю ОФЗ не менее 35% как «якорь» надёжности

Довести средневзвешенную дюрацию портфеля до 3.0 лет (иммунизация)

Ограничить концентрацию: ≤ 10% на корпоративного эмитента, ≤ 2 бумаги из одного сектора

В рамках выполненных задач 1–4 — максимизировать YTM (решение Solver/SLSQP)

Итог: YTM 19.73%, дюрация 2.95 лет, 7 секторов, 10 позиций, HHI = 0.109 — умеренная концентрация.

